# Problem Sheet 3 — Question 4(a)

A measurement of four parameters gives

| | value |
|---|---|
| $x^+$ | $-9.3 \pm 8.2$ |
| $y^+$ | $-1.3 \pm 8.4$ |
| $x^-$ | $\phantom{-}5.7 \pm 8.2$ |
| $y^-$ | $\phantom{-}6.5 \pm 8.3$ |

with a known correlation matrix. Assuming the observations are multivariate normally distributed, use the **maximum likelihood (ML)** method to estimate the values, uncertainties and correlations of $(\gamma, r, \delta)$, where

$$x^\pm = r\cos(\delta \pm \gamma), \qquad y^\pm = r\sin(\delta \pm \gamma).$$

## Setup
Install `iminuit` if you don't have it (needed on Google Colab).

In [1]:
%pip install -q iminuit

In [2]:
import numpy as np
from iminuit import Minuit

## The measurements

The covariance matrix is built from the uncertainties and correlations: $V_{ij} = \rho_{ij}\,\sigma_i\,\sigma_j$. Order: $x^+, y^+, x^-, y^-$.

In [3]:
obs = np.array([-9.3, -1.3, 5.7, 6.5])
err = np.array([8.2, 8.4, 8.2, 8.3])

corr = np.array([
    [ 1.00, -0.10, -0.05,  0.10],
    [-0.10,  1.00,  0.10, -0.05],
    [-0.05,  0.10,  1.00,  0.10],
    [ 0.10, -0.05,  0.10,  1.00],
])

cov = corr * np.outer(err, err)
cov_inv = np.linalg.inv(cov)

## The likelihood

For a multivariate normal distribution,

$$-2\ln L = (\mathbf{x}_\text{obs} - \mathbf{x}_\text{model})^T\, V^{-1}\, (\mathbf{x}_\text{obs} - \mathbf{x}_\text{model}) + \text{const},$$

so maximising $L$ is the same as minimising this $\chi^2$. We minimise $-\ln L = \tfrac{1}{2}\chi^2$.

In [4]:
def model(gamma, r, delta):
    """Predicted (x+, y+, x-, y-) for given parameters (angles in radians)."""
    return np.array([
        r * np.cos(delta + gamma),
        r * np.sin(delta + gamma),
        r * np.cos(delta - gamma),
        r * np.sin(delta - gamma),
    ])


def nll(gamma, r, delta):
    """Negative log-likelihood (up to a constant) = 0.5 * chi^2."""
    d = obs - model(gamma, r, delta)
    return 0.5 * d @ cov_inv @ d

## The fit

The sines and cosines make the likelihood have several separate minima, so we start the minimiser from a grid of starting points and keep the best result. $r$ is a magnitude, so it is limited to $r \ge 0$.

`errordef = 0.5` tells Minuit we are minimising $-\ln L$ (rather than $\chi^2$), so the $1\sigma$ uncertainties come from where $-\ln L$ rises by 0.5. HESSE then computes the uncertainties and correlations from the curvature at the minimum.

In [5]:
best = None
for g0 in np.linspace(0, np.pi, 7):
    for d0 in np.linspace(-np.pi, np.pi, 9):
        m = Minuit(nll, gamma=g0, r=5.0, delta=d0)
        m.errordef = Minuit.LIKELIHOOD   # 0.5
        m.limits["r"] = (0, None)
        m.migrad()
        if m.valid and (best is None or m.fval < best.fval):
            best = m

m = best
m.hesse()

┌─────────────────────────────────────────────────────────────────────────┐
│                                Migrad                                   │
├──────────────────────────────────┬──────────────────────────────────────┤
│ FCN = 0.001894                   │              Nfcn = 123              │
│ EDM = 9.72e-09 (Goal: 0.0001)    │                                      │
├──────────────────────────────────┼──────────────────────────────────────┤
│          Valid Minimum           │   Below EDM threshold (goal x 10)    │
├──────────────────────────────────┼──────────────────────────────────────┤
│      No parameters at limit      │           Below call limit           │
├──────────────────────────────────┼──────────────────────────────────────┤
│             Hesse ok             │         Covariance accurate          │
└──────────────────────────────────┴──────────────────────────────────────┘
┌───┬───────┬───────────┬───────────┬────────────┬────────────┬─────────┬─────────┬───────┐
│   │ Name  │   Value   │ Hesse Err │ Minos Err- │ Minos Err+ │ Limit-  │ Limit+  │ Fixed │
├───┼───────┼───────────┼───────────┼────────────┼────────────┼─────────┼─────────┼───────┤
│ 0 │ gamma │   -1.9    │    0.6    │            │            │         │         │       │
│ 1 │ r     │     9     │     6     │            │            │    0    │         │       │
│ 2 │ delta │   -1.1    │    0.7    │            │            │         │         │       │
└───┴───────┴───────────┴───────────┴────────────┴────────────┴─────────┴─────────┴───────┘
┌───────┬───────────────────┐
│       │ gamma     r delta │
├───────┼───────────────────┤
│ gamma │ 0.355  -0.0   0.0 │
│     r │  -0.0  33.3  -0.4 │
│ delta │   0.0  -0.4 0.455 │
└───────┴───────────────────┘

## Results

$\gamma$ and $\delta$ are only defined modulo $360^\circ$, and $(\gamma, \delta) \to (\gamma + 180^\circ, \delta + 180^\circ)$ gives an identical prediction. We quote $\gamma \in [0^\circ, 180^\circ)$ and $\delta \in (-180^\circ, 180^\circ]$.

In [6]:
gamma, r, delta = m.values["gamma"], m.values["r"], m.values["delta"]
if gamma % (2 * np.pi) >= np.pi:
    gamma, delta = gamma - np.pi, delta - np.pi
gamma = gamma % np.pi
delta = (delta + np.pi) % (2 * np.pi) - np.pi

s_gamma, s_r, s_delta = m.errors["gamma"], m.errors["r"], m.errors["delta"]
deg = 180 / np.pi

print(f"gamma = {gamma * deg:6.1f} ± {s_gamma * deg:4.1f} degrees")
print(f"r     = {r:6.2f} ± {s_r:4.2f}")
print(f"delta = {delta * deg:6.1f} ± {s_delta * deg:4.1f} degrees")
print(f"\nchi^2 at minimum = {2 * m.fval:.3f}  (4 observables - 3 parameters = 1 dof)")

gamma =   69.6 ± 34.2 degrees
r     =   9.04 ± 5.76
delta =  118.5 ± 38.7 degrees

chi^2 at minimum = 0.004  (4 observables - 3 parameters = 1 dof)


### Correlation matrix

In [7]:
names = ["gamma", "r", "delta"]
corr_fit = np.array(m.covariance.correlation())
print("        " + "".join(f"{n:>9}" for n in names))
for n, row in zip(names, corr_fit):
    print(f"{n:<8}" + "".join(f"{v:9.3f}" for v in row))

            gamma        r    delta
gamma       1.000   -0.009    0.048
r          -0.009    1.000   -0.101
delta       0.048   -0.101    1.000


### Check: does the fitted model reproduce the measurements?

In [8]:
pred = model(m.values["gamma"], m.values["r"], m.values["delta"])
for name, o, p in zip(["x+", "y+", "x-", "y-"], obs, pred):
    print(f"{name}: measured {o:5.1f}   fitted {p:6.2f}")

x+: measured  -9.3   fitted  -8.95
y+: measured  -1.3   fitted  -1.27
x-: measured   5.7   fitted   5.94
y-: measured   6.5   fitted   6.82


## Comments

* The $\chi^2$ at the minimum is almost zero: with 4 measurements and 3 parameters there is only 1 degree of freedom, so the model can match the data very closely.
* $r$ is only about $1.5\sigma$ from zero, so the likelihood is not very parabolic and the HESSE uncertainties should be treated as approximate. This is explored with the profile likelihood scan in part (b).
* The correlations between $\gamma$, $r$ and $\delta$ are small.